# Yahoo <-> EODHD Konsistenzcheck fuer die NIS/CUSUM/SMA-EMA-Pipeline
**Analog zum Paper-2-Befund vom 30.06.2026 ("Yahoo-Demo-Layer: Quellen-Konsistenz bewiesen"),
diesmal fuer unsere eigenen Groessen, nicht Paper 2s Momente.**

**Zweck:** Zeigen, dass die im oeffentlichen Demo-Repo verwendeten Yahoo-Daten (15 Ticker) zu
denselben NIS-/CUSUM-/SMA-EMA-Ergebnissen fuehren wie die lizenzierte EODHD-Quelle -- damit der
Transparenz-Satz aus Paper 2 uebertragen werden kann: *"Glaub uns nicht, bau es selbst nach."*

**Wichtig, wie beim 30.06.-Befund:** die Kernfunktionen werden WOERTLICH aus `src/nis_core.py`
importiert, nicht aus dem Gedaechtnis nachgebaut -- "Nachrechnen statt Erinnern, auch beim Code."

**Ergebnis dieses Notebooks:** eine Entscheidungstabelle pro Ticker (Preiskorrelation, NIS-
Korrelation, CUSUM-Korrelation, ob SMA/EMA-Kreuzungsdaten uebereinstimmen) -- kein Blindvertrauen,
sondern eine harte Zahl pro Groesse.

## 1. Daten laden: EODHD (Quelle der Wahrheit) und Yahoo-Demo

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
# WOERTLICH aus dem echten Modul importieren -- nicht nachbauen (Lehre vom 30.06.)
from nis_core import (directional_change, sma_ema_crossings, kalman_plain_vanilla,
                       cusum_from_std_innovations, split_into_continuous_segments)

EODHD_PATH = Path.home() / "Paper4" / "data" / "eodhd_1135_cleaned.parquet"
YAHOO_PATH = REPO_ROOT / "data" / "demo_panel.parquet"

if not EODHD_PATH.exists():
    raise FileNotFoundError(f"{EODHD_PATH} nicht gefunden -- bitte Pfad pruefen.")
if not YAHOO_PATH.exists():
    raise FileNotFoundError(f"{YAHOO_PATH} nicht gefunden -- erst download_demo_data.py laufen lassen.")

eodhd = pd.read_parquet(EODHD_PATH)
eodhd['date'] = pd.to_datetime(eodhd['date'])
yahoo = pd.read_parquet(YAHOO_PATH)
yahoo['date'] = pd.to_datetime(yahoo['date'])

print(f"EODHD: {len(eodhd)} Zeilen, {eodhd['ticker'].nunique()} Ticker")
print(f"Yahoo: {len(yahoo)} Zeilen, {yahoo['ticker'].nunique()} Ticker")

## 2. Symbol-Mapping: Yahoo-Ticker -> EODHD-Ticker

In [ ]:
# Unsere 15 Demo-Ticker sind alle US-Titel -- EODHD haengt ".US" an, Yahoo nicht.
# (Fuer eine spaetere Erweiterung auf internationale Ticker: gleiche Mapping-Tabelle
#  wie im Paper-2-Diagnose-Notebook verwenden.)
def to_eodhd_symbol(yahoo_tk):
    return yahoo_tk + ".US"

demo_tickers_yahoo = sorted(yahoo['ticker'].unique())
mapping = {tk: to_eodhd_symbol(tk) for tk in demo_tickers_yahoo}
missing_in_eodhd = [y for y, e in mapping.items() if e not in eodhd['ticker'].unique()]
print(f"Demo-Ticker: {len(demo_tickers_yahoo)}")
if missing_in_eodhd:
    print(f"WARNUNG -- nicht im EODHD-Panel gefunden (werden uebersprungen): {missing_in_eodhd}")
else:
    print("Alle Demo-Ticker im EODHD-Panel gefunden.")

## 3. Pro Ticker: Kurskorrelation + NIS/CUSUM-Korrelation + SMA/EMA-Kreuzungsvergleich

In [ ]:
DC_DELTA = 0.10
Q_PARAM = 1e-4
R_CONST = 1e-4
CUSUM_WINDOW = 20
CROSS_DATE_TOLERANCE_DAYS = 2  # SMA/EMA-Kreuzung gilt als "gleich", wenn Datum um <= das abweicht

rows = []
for y_tk in demo_tickers_yahoo:
    e_tk = mapping[y_tk]
    if e_tk not in eodhd['ticker'].unique():
        continue

    y_sub = yahoo[yahoo['ticker'] == y_tk].sort_values('date').reset_index(drop=True)
    e_sub = eodhd[eodhd['ticker'] == e_tk].sort_values('date').reset_index(drop=True)

    rec = dict(ticker=y_tk, eodhd_ticker=e_tk, yahoo_rows=len(y_sub), eodhd_rows=len(e_sub))

    # Gemeinsame Handelstage
    merged_px = y_sub[['date', 'close']].merge(e_sub[['date', 'close']], on='date',
                                                 suffixes=('_yahoo', '_eodhd'))
    rec['common_days'] = len(merged_px)
    if len(merged_px) < 100:
        rec['status'] = 'ZU_WENIG_GEMEINSAME_TAGE'
        rows.append(rec)
        continue

    # Preis-Korrelation (sollte nahe 1 sein, da beide adjusted_close-basiert)
    price_corr = np.corrcoef(merged_px['close_yahoo'], merged_px['close_eodhd'])[0, 1]
    rec['price_corr'] = price_corr

    # Split-Sonde: groesster Einzeltag-Sprung im Verhaeltnis Yahoo/EODHD (wie 30.06.-Diagnose)
    ratio = (merged_px['close_yahoo'] / merged_px['close_eodhd']).values
    ratio = ratio[np.isfinite(ratio) & (ratio > 0)]
    if len(ratio) > 10:
        log_ratio_diff = np.diff(np.log(ratio))
        j = int(np.argmax(np.abs(log_ratio_diff)))
        rec['max_ratio_jump'] = float(np.abs(log_ratio_diff[j]))
    else:
        rec['max_ratio_jump'] = np.nan

    # NIS und CUSUM auf beiden Quellen unabhaengig berechnen, dann auf gemeinsamen Tagen vergleichen
    def compute_nis_cusum(sub):
        prices = sub['close'].values.astype(float)
        dates = sub['date'].values
        segments = split_into_continuous_segments(dates, max_gap_days=10)
        nis_full = np.full(len(prices), np.nan)
        cusum_full = np.full(len(prices), np.nan)
        for s, e in segments:
            seg_prices = prices[s:e]
            if len(seg_prices) < 30:
                continue
            log_p = np.log(seg_prices)
            nis, std_innov = kalman_plain_vanilla(log_p, Q=Q_PARAM, R=R_CONST)
            cusum = cusum_from_std_innovations(std_innov, window=CUSUM_WINDOW)
            nis_full[s:e] = nis
            cusum_full[s:e] = cusum
        return pd.DataFrame({'date': dates, 'nis': nis_full, 'cusum': cusum_full})

    y_stats = compute_nis_cusum(y_sub)
    e_stats = compute_nis_cusum(e_sub)
    merged_stats = y_stats.merge(e_stats, on='date', suffixes=('_yahoo', '_eodhd')).dropna()

    if len(merged_stats) > 50:
        rec['nis_corr'] = np.corrcoef(merged_stats['nis_yahoo'], merged_stats['nis_eodhd'])[0, 1]
        rec['cusum_corr'] = np.corrcoef(merged_stats['cusum_yahoo'], merged_stats['cusum_eodhd'])[0, 1]
    else:
        rec['nis_corr'] = np.nan
        rec['cusum_corr'] = np.nan

    # SMA/EMA-Kreuzungsdaten vergleichen: wie viele Kreuzungen stimmen (+-Toleranz) ueberein?
    y_cross_idx = sma_ema_crossings(y_sub['close'].values.astype(float))
    e_cross_idx = sma_ema_crossings(e_sub['close'].values.astype(float))
    y_cross_dates = set(pd.Timestamp(d).normalize() for d in y_sub['date'].values[y_cross_idx])
    e_cross_dates = set(pd.Timestamp(d).normalize() for d in e_sub['date'].values[e_cross_idx])

    def date_matches(d, date_set, tol_days):
        return any(abs((d - other).days) <= tol_days for other in date_set)

    matched = sum(1 for d in y_cross_dates if date_matches(d, e_cross_dates, CROSS_DATE_TOLERANCE_DAYS))
    rec['yahoo_crossings'] = len(y_cross_dates)
    rec['eodhd_crossings'] = len(e_cross_dates)
    rec['crossings_matched'] = matched
    rec['crossing_match_rate'] = matched / max(len(y_cross_dates), 1)

    mc = rec.get('nis_corr', np.nan)
    rec['status'] = ('OK' if (mc == mc and mc > 0.99) else
                     'BORDERLINE' if (mc == mc and mc > 0.95) else
                     'PRUEFEN')
    rows.append(rec)

diag = pd.DataFrame(rows)
print(f"{len(diag)} Ticker verarbeitet.")

## 4. Zusammenfassung

In [ ]:
print("Status-Verteilung:")
print(diag['status'].value_counts().to_string())

print(f"\nDurchschnittliche Preis-Korrelation: {diag['price_corr'].mean():.6f}")
print(f"Durchschnittliche NIS-Korrelation:   {diag['nis_corr'].mean():.6f}")
print(f"Durchschnittliche CUSUM-Korrelation: {diag['cusum_corr'].mean():.6f}")
print(f"Durchschnittliche Crossing-Match-Rate: {100*diag['crossing_match_rate'].mean():.1f}%")

print("\nAlle Ticker im Detail:")
cols = ['ticker', 'eodhd_ticker', 'common_days', 'price_corr', 'max_ratio_jump',
        'nis_corr', 'cusum_corr', 'crossing_match_rate', 'status']
print(diag[cols].to_string(index=False))

print("\n--- Nicht-OK-Ticker (brauchen Aufmerksamkeit) ---")
probs = diag[diag['status'] != 'OK']
if len(probs):
    print(probs[cols].to_string(index=False))
else:
    print("Keine -- alle Ticker sind konsistent (NIS-Korrelation > 0.99).")

OUT_PATH = Path.home() / "Paper4" / "notebook_outputs" / "yahoo_eodhd_nis_consistency.parquet"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
diag.to_parquet(OUT_PATH)
print(f"\nGespeichert: {OUT_PATH}")